In [1]:
pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 39.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
#Use a pipeline as a high-level helper
from transformers import pipeline


pipe = pipeline("text-generation", model="ibm-granite/granite-3.3-2b-instruct")
messages = [
	{"role": "user", "content": "Who are you?"},
]
pipe(messages)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.'}]}]

In [3]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM


tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
model= AutoModelForCausalLM.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
messages = [
	{"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
  messages,
  add_generation_prompt=True,
  tokenize=True,
  return_dict=True,
  return_tensors="pt",
).to(model.device)


outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.<|end_of_text|>


In [4]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 19.1 MB/s eta 0:00:00


In [5]:
!pip install gradio transformers torch black autopep8 reportlab requests

In [6]:
import sqlite3

In [7]:
!pip install -q gradio reportlab requests

import gradio as gr
from datetime import datetime
from pathlib import Path
import re
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors

# =====================================================
# DATABASE SCHEMA
# =====================================================
SCHEMA = {
    "users": ["id", "email", "signup_date", "age", "country", "status"],
    "products": ["id", "name", "price", "category", "stock"],
    "transactions": ["id", "user_id", "amount", "date", "type"],
}

DANGEROUS_KEYWORDS = ["DROP", "DELETE", "ALTER", "TRUNCATE", "INSERT", "UPDATE", "EXEC"]
query_history = []

# =====================================================
# SECURITY CHECK
# =====================================================
def is_safe(text):
    return not any(word in text.upper() for word in DANGEROUS_KEYWORDS)

# =====================================================
# TABLE DETECTION
# =====================================================
def detect_table(query):
    q = query.lower()
    if any(w in q for w in ["user", "customer"]): return "users"
    if any(w in q for w in ["product", "item"]): return "products"
    if any(w in q for w in ["transaction", "payment"]): return "transactions"
    return None

# =====================================================
# CONDITION EXTRACTION
# =====================================================
def extract_conditions(query):
    q = query.lower()
    conditions = []
    if "last 2 days" in q: conditions.append("DATE(signup_date) >= DATE('now','-2 day')")
    if "last 5 days" in q: conditions.append("DATE(date) >= DATE('now','-5 day')")
    if "last month" in q: conditions.append("DATE(signup_date) >= DATE('now','-1 month')")
    amount = re.search(r"greater than (\d+)", q)
    if amount: conditions.append(f"amount > {amount.group(1)}")
    return conditions

# =====================================================
# SQL ENGINE
# =====================================================
def generate_sql(user_query):
    if not is_safe(user_query): return "⚠ Unsafe query detected!"
    table = detect_table(user_query)
    if not table: return "⚠ Could not detect table."
    conditions = extract_conditions(user_query)
    sql = f"SELECT * FROM {table}"
    if conditions: sql += " WHERE " + " AND ".join(conditions)
    sql += ";"
    query_history.append(sql)
    return sql

# =====================================================
# EXPORT FILES
# =====================================================
def generate_pdf(user_query, sql_query):
    filename = "SQL_Report.pdf"
    doc = SimpleDocTemplate(filename)
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(name="TitleStyle", parent=styles["Heading1"], fontSize=18, textColor=colors.darkblue)
    story = [Paragraph("SQL QUERY GENERATOR REPORT", title_style), Spacer(1, 0.3*inch),
             Paragraph(f"Timestamp: {datetime.now()}", styles["Normal"]), Spacer(1, 0.2*inch),
             Paragraph(f"User Query: {user_query}", styles["Normal"]), Spacer(1, 0.2*inch),
             Paragraph(f"Generated SQL: {sql_query}", styles["Normal"])]
    doc.build(story)
    return filename

def save_text(user_query, sql_query):
    filename = "SQL_Report.txt"
    content = f"Timestamp: {datetime.now()}\n\nUser Query: {user_query}\nGenerated SQL: {sql_query}"
    Path(filename).write_text(content)
    return filename

# =====================================================
# MAIN PROCESS
# =====================================================
def process_query(user_query):
    sql = generate_sql(user_query)
    if "⚠" in sql:
        return f"<div style='color:red;'>{sql}</div>", None, None

    pdf_file = generate_pdf(user_query, sql)
    text_file = save_text(user_query, sql)

    table_name = detect_table(user_query) or "transactions"

    # The "double square" copy icon SVG
    copy_icon_svg = """<svg xmlns="http://www.w3.org/2000/svg" width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" style="opacity: 0.8;"><rect x="9" y="9" width="13" height="13" rx="2" ry="2"></rect><path d="M5 15H4a2 2 0 0 1-2-2V4a2 2 0 0 1 2-2h9a2 2 0 0 1 2 2v1"></path></svg>"""

    styled_sql = f"""
    <div style='display: flex; flex-direction: column; gap: 10px; width: 100%;'>

        <div style='display: flex; flex-direction: column; align-items: flex-end; gap: 5px; width: 100%;'>
            <div style='background-color: #374151; color: #ffffff; padding: 10px 15px; border-radius: 12px; font-size: 14px; max-width: 80%; border-bottom-right-radius: 2px;'>
                {user_query}
            </div>
            <button onclick="navigator.clipboard.writeText(`{sql}`)" style='cursor: pointer; background: none; border: none; color: #6b7280; padding: 2px;'>
                {copy_icon_svg}
            </button>
        </div>

        <div style='background-color: #111827; border: 1px solid #1f2937; border-radius: 8px; padding: 15px; width: 100%;'>
            <div style='display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px;'>
                <span style='color: #4ade80; font-weight: bold; font-size: 13px; display: flex; align-items: center; gap: 6px;'>
                    <span style="background:#4ade80; color:#111827; border-radius:3px; padding:0 2px; font-size: 10px;">✓</span> SQL GENERATED
                </span>
            </div>

            <div style="background-color: #1a1a1a; padding: 12px; border-radius: 6px; border: 1px solid #374151; position: relative; min-height: 40px; display: flex; align-items: center;">
                <code style="font-family: monospace; color: #f472b6; font-size: 14px; padding-right: 30px; word-break: break-all;">{sql}</code>
                <button onclick="navigator.clipboard.writeText(`{sql}`)" style='position: absolute; top: 10px; right: 10px; cursor: pointer; background: none; border: none; color: #6b7280;'>
                    {copy_icon_svg}
                </button>
            </div>

            <div style='margin-top: 12px; display: flex; align-items: center; gap: 8px; color: #9ca3af; font-size: 12px;'>
                <span style="opacity: 0.8;">📄</span>
                <span style="color: #4ade80; font-size: 10px;">✓</span>
                <span>Generated from table '{table_name}' with columns: *</span>
            </div>
        </div>
    </div>
    """
    return styled_sql, pdf_file, text_file

    pdf_file = generate_pdf(user_query, sql)
    text_file = save_text(user_query, sql)

    table_name = detect_table(user_query) or "transactions"

    # SVG for the double-square copy icon
    copy_icon_svg = """<svg xmlns="http://www.w3.org/2000/svg" width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2" ry="2"></rect><path d="M5 15H4a2 2 0 0 1-2-2V4a2 2 0 0 1 2-2h9a2 2 0 0 1 2 2v1"></path></svg>"""

    styled_sql = f"""
    <div style='display: flex; flex-direction: column; gap: 10px; width: 100%; font-family: sans-serif;'>
        <div style='align-self: flex-end; background-color: #374151; color: white; padding: 10px 15px; border-radius: 12px; font-size: 14px; max-width: 80%;'>
            {user_query}
        </div>

        <div style='background-color: #111827; border: 1px solid #1f2937; border-radius: 8px; padding: 15px; width: 100%; position: relative;'>
            <div style='display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;'>
                <span style='color: #4ade80; font-weight: bold; font-size: 13px; display: flex; align-items: center; gap: 5px;'>
                    <span style="background:#4ade80; color:#111827; border-radius:3px; padding:0 2px;">✓</span> SQL GENERATED
                </span>
            </div>

            <div style="background-color: #1a1a1a; padding: 12px; border-radius: 6px; border: 1px solid #374151; position: relative;">
                <code style="font-family: monospace; color: #f472b6; font-size: 14px;">{sql}</code>
                <button onclick="navigator.clipboard.writeText(`{sql}`)" style='position: absolute; top: 8px; right: 8px; cursor: pointer; background: none; border: none; color: #9ca3af;'>
                    {copy_icon_svg}
                </button>
            </div>

            <div style='margin-top: 12px; display: flex; align-items: center; gap: 8px; color: #9ca3af; font-size: 12px;'>
                <span>📄</span>
                <span style="color: #4ade80;">✓</span>
                <span>Generated from table '{table_name}' with columns: *</span>
            </div>
        </div>

        <button onclick="navigator.clipboard.writeText(`{sql}`)" style='align-self: flex-start; cursor: pointer; background: none; border: none; color: #6b7280; padding: 5px;'>
            {copy_icon_svg}
        </button>
    </div>
    """
    return styled_sql, pdf_file, text_file

def clear_history():
    query_history.clear()
    return "", None, None

# =====================================================
# CSS STYLING
# =====================================================
custom_css = """
body, .gradio-container { background-color: #0b0f1a !important; color: white !important; }
.panel-style { border-radius: 8px; border: 1px solid #1e293b !important; background: #0f172a !important; padding: 15px; margin-bottom: 10px; }
.sql-card { background: #111827; border: 1px solid #1f2937; border-radius: 8px; padding: 12px; width: fit-content; min-width: 300px; }
.sql-header { display: flex; justify-content: space-between; font-size: 0.75em; color: #9ca3af; margin-bottom: 8px; font-weight: bold; }
.sql-code { font-family: 'Courier New', monospace; color: #f472b6; font-size: 1.1em; margin-bottom: 8px; }
.sql-footer { font-size: 0.7em; color: #6b7280; }
.blue-btn { background: #2563eb !important; color: white !important; border: none !important; }
textarea, input { background-color: #111827 !important; color: white !important; border: 1px solid #1e293b !important; }
"""

# =====================================================
# GRADIO UI LAYOUT
# =====================================================
with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# 🔍 SQL Query Generator")
    gr.Markdown("### Natural Language → SQLite SQL")
    gr.Markdown("Convert plain English requests into safe, production-ready SQL queries!")

    with gr.Row():
        # LEFT COLUMN
        with gr.Column(scale=4):
            with gr.Column(variant="panel", elem_classes="panel-style"):
                gr.Markdown("💬 **Query Conversion**")
                output_sql = gr.HTML()

            user_input = gr.Textbox(label="Natural Language Request", placeholder="e.g., show me all users...")
            generate_btn = gr.Button("Generate SQL", variant="primary", elem_classes="blue-btn")

        # RIGHT COLUMN (3 SECTIONS)
        with gr.Column(scale=1):
            with gr.Column(variant="panel", elem_classes="panel-style"):
                gr.Markdown("🕹️ **Controls**")
                clear_btn = gr.Button("Clear History")

            with gr.Column(variant="panel", elem_classes="panel-style"):
                gr.Markdown("💾 **Export**")
                pdf_btn = gr.Button("Download PDF", elem_classes="blue-btn")
                txt_btn = gr.Button("Download TXT", elem_classes="blue-btn")

            with gr.Column(variant="panel", elem_classes="panel-style"):
                gr.Markdown("📂 **Files**")
                pdf_file = gr.File(label="PDF File")
                txt_file = gr.File(label="TXT File")

    # Click Events
    generate_btn.click(process_query, inputs=[user_input], outputs=[output_sql, pdf_file, txt_file])
    clear_btn.click(clear_history, outputs=[output_sql, pdf_file, txt_file])

demo.launch()

/tmp/ipykernel_318/3847362781.py:203: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a051c5e07576bfe45.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
